# 6.2 图像卷积

二维互相关运算是卷积层的核心计算。深度学习框架中的“卷积”通常实现为互相关：卷积核在输入上滑动，对窗口元素逐项相乘再求和。


In [1]:
import torch
from torch import nn

## 互相关运算


In [2]:
def corr2d(X, K):
    """计算二维互相关运算。"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y


X = torch.tensor([[0.0, 1.0, 2.0],
                  [3.0, 4.0, 5.0],
                  [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0],
                  [2.0, 3.0]])
print(corr2d(X, K))


tensor([[19., 25.],
        [37., 43.]])


## 自定义二维卷积层

卷积层的可学习参数包括卷积核权重和偏置。前向传播时对输入做互相关，再加上偏置。


In [3]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias


## 图像中目标的边缘检测

构造一张中间为 0、两侧为 1 的简单图像，再用 `[1, -1]` 卷积核检测垂直边缘。


In [4]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
print(X)

K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K)
print(Y)


tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])
tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])


如果把图像转置，原来的垂直边缘变成水平边缘，此时同一个卷积核就无法检测出来。


In [5]:
print(corr2d(X.t(), K))


tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


## 学习卷积核

使用 `nn.Conv2d` 学习上面的边缘检测卷积核。输入形状需要写成 `(批量大小, 通道数, 高度, 宽度)`。


In [6]:
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)
X_train = X.reshape((1, 1, 6, 8))
Y_train = Y.reshape((1, 1, 6, 7))

for i in range(10):
    Y_hat = conv2d(X_train)
    loss = (Y_hat - Y_train) ** 2
    conv2d.zero_grad()
    loss.sum().backward()
    conv2d.weight.data[:] -= 0.03 * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i + 1}, loss {loss.sum():.3f}')

print('learned kernel:', conv2d.weight.data.reshape((1, 2)))


epoch 2, loss 9.639
epoch 4, loss 2.553
epoch 6, loss 0.812
epoch 8, loss 0.293
epoch 10, loss 0.114
learned kernel: tensor([[ 1.0216, -0.9534]])


训练后的权重会接近 `[1, -1]`，说明卷积核可以通过数据自动学习到图像中的局部模式。
